In [4]:
import numpy as np

STUDENT = "Нарижна А.О."
VARIANT = 9
STUDENT_ID = 450 
SEED = 1000 * VARIANT + STUDENT_ID

rng = np.random.default_rng(SEED)

In [3]:
import textwrap

with open("source.txt", encoding="utf-8") as f:
    source = " ".join(f.read().split())

corpus = textwrap.wrap(
    source, width=65,
    break_long_words=False,
    break_on_hyphens=False
)

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(corpus))

import os
import hashlib
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageDraw, ImageFont

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

FONTS = ["C:/Windows/Fonts/arial.ttf"]

with open("corpus.txt", encoding="utf-8") as f:
    corpus = [s.strip() for s in f if 20 <= len(s.strip()) <= 70]

if len(corpus) < 40:
    raise ValueError("Потрібно щонайменше 40 рядків довжиною 20–70 символів у корпусі.")

lines = [str(s) for s in rng.choice(corpus, size=40, replace=False)]

def render_line(text, font_path, size=32, pad=12):
    font = ImageFont.truetype(font_path, size)
    l, t, r, b = font.getbbox(text)
    img = Image.new("L", (r - l + 2 * pad, b - t + 2 * pad), 255)
    ImageDraw.Draw(img).text((pad - l, pad - t), text, font=font, fill=0)
    return np.array(img)

def degrade(img, kind, level, rng):
    if kind == "font_height":
        # Масштабуємо текст без білих полів до заданої висоти.
        ys, xs = np.where(img < 255)
        text_img = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        h, w = text_img.shape
        resized = cv2.resize(
            text_img,
            (max(1, round(w * level / h)), level),
            interpolation=cv2.INTER_AREA if level < h else cv2.INTER_CUBIC
        )
        return cv2.copyMakeBorder(resized, 12, 12, 12, 12,
                                  cv2.BORDER_CONSTANT, value=255)
    return img

FACTOR, LEVELS = "font_height", [40, 32, 24, 18, 14, 12, 10]
records = []

for i, text in enumerate(lines):
    font = FONTS[i % len(FONTS)]
    clean = render_line(text, font)

    for lvl in LEVELS:
        path = f"data/{i:03d}_{FACTOR}_{lvl}.png"
        cv2.imwrite(path, degrade(clean, FACTOR, lvl, rng))
        records.append({
            "id": i, "level": lvl, "path": path,
            "gt": text, "font": os.path.basename(font)
        })

ds = pd.DataFrame(records)
ds.to_csv("data/ground_truth.csv", index=False)

with open("data/ground_truth.csv", "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()

print("Записів:", len(ds), "| SHA-256 еталону:", sha)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(LEVELS), 1, figsize=(7, 0.75 * len(LEVELS)))

for ax, lvl in zip(axes, LEVELS):
    img = cv2.imread(f"data/000_{FACTOR}_{lvl}.png", cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Висота = {lvl} px", fontsize=8, loc="left")
    ax.axis("off")

fig.suptitle(f"{STUDENT}, варіант {VARIANT}, seed={SEED}", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig("figures/fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()


NameError: name 'rng' is not defined

In [2]:
%pip install datasets

  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached pyarrow-26.0.0-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-4.0.2-cp314-cp314-win_amd64.whl.metadata (18 kB)
  Using cached multiprocess-0.70.19-py314-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached aiohttp-3.14.5-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.7.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata 


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from datasets import load_dataset
import os
import pandas as pd

hf = load_dataset("Teklia/IAM-line", split="test")
print(hf)
print(hf.features)

os.makedirs("data/iam", exist_ok=True)
records_iam = []

for i in range(100):
    sample = hf[i]
    path = f"data/iam/{i:03d}.png"
    sample["image"].save(path)
    records_iam.append({
        "id": i,
        "dataset": "Teklia/IAM-line",
        "split": "test",
        "path": path,
        "gt": sample["text"]
    })

ds_iam = pd.DataFrame(records_iam)
ds_iam.to_csv("data/iam/ground_truth.csv", index=False)
print("Збережено рядків IAM:", len(ds_iam))
print("Еталон першого рядка:", ds_iam.loc[0, "gt"])


c:\Users\Аліна\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Аліна\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Аліна\.cache\huggingface\hub\datasets--Teklia--IAM-line. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admini

Dataset({
    features: ['image', 'text'],
    num_rows: 2915
})
{'image': Image(mode=None, decode=True), 'text': Value('string')}
Збережено рядків IAM: 100
Еталон першого рядка: assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .


## Пункт 8. Гіпотези до запуску розпізнавання

Дата формулювання: 08.11.2026. Варіант 9, SEED = 9450.

Наведені значення є прогнозами, а не результатами вимірювань. Для перевірки використовуватиметься середній CER по зразках з однаковими правилами нормалізації тексту. Для синтетичного українського тексту Tesseract працюватиме з мовою ukr, для IAM та англомовних нотаток — eng.

1. **Вплив висоти тексту.** На синтетичному наборі без попередньої обробки Tesseract матиме середній CER не більше 10 % за висоти тексту 32 px, а за висоти 10 px — понад 10 %. Причина прогнозу: зменшення літер призводить до втрати дрібних елементів і злиття штрихів. Гіпотеза підтвердиться, якщо виконаються обидві нерівності.

2. **Вплив попередньої обробки.** На синтетичних рядках висотою 10 px послідовність із прикладу методички — збільшення у 2 рази, медіанний фільтр і бінаризація Оцу — зменшить середній CER Tesseract щонайменше на 5 відсоткових пунктів порівняно з тими самими рядками без обробки. Причина прогнозу: збільшення полегшить розпізнавання дрібних літер, а бінаризація виділить текст. Критерій: CER_raw − CER_prep ≥ 0,05.

3. **Власні фото проти синтетичних зображень.** Без попередньої обробки середній CER Tesseract на трьох власних фотографіях української рукописної сторінки буде щонайменше вдвічі вищим, ніж на синтетичних українських рядках висотою 32 px. Причина прогнозу: індивідуальний почерк, тіні та перспектива ускладнюють розпізнавання порівняно з друкованим текстом. Критерій: CER_фото ≥ 2 × CER_синтетика; обидва значення буде наведено окремо. Англомовні нотатки до цієї перевірки не включаються.

Після експериментів для кожної гіпотези буде наведено фактичні значення, оцінку «підтверджено» або «спростовано» та пояснення. Спростування гіпотези також є результатом експерименту.
